In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, EsmTokenizer, AutoModelForMaskedLM
import pandas as pd
from tqdm import tqdm
import os

# --- 1. Config ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ESM_PATH = "facebook/esm2_t6_8M_UR50D"
BATCH_SIZE = 64
EPOCHS = 5  #coverage quickly,so less epoch
LR = 1e-4

# because the decoder converage too fast,we add some noise to make adaptation
NOISE_LEVEL = 0.2

SAVE_PATH = "trained_models_cfg/decoder_robust.pth"

# --- 2. Dataset ---
class DecoderDataset(Dataset):
    def __init__(self, csv_path, tokenizer, esm_model, device):
        df = pd.read_csv(csv_path)
        col = 'sequence' if 'sequence' in df.columns else 'SEQUENCE'
        self.seqs = df[col].dropna().tolist()
        self.tokenizer = tokenizer
        self.esm_model = esm_model.to(device).eval()
        self.device = device

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        with torch.no_grad():
            tokens = self.tokenizer(seq, return_tensors="pt", padding='max_length', max_length=72, truncation=True)
            input_ids = tokens['input_ids'].to(self.device)
            outputs = self.esm_model(input_ids)
            embeddings = outputs.last_hidden_state.squeeze(0) # [72, 320]
            labels = input_ids.squeeze(0) # [72]

        return embeddings.cpu(), labels.cpu()

# --- 3. Training ---
def main():
    if not os.path.exists("trained_models_cfg"):
        os.makedirs("trained_models_cfg")

    print("Loading ESM Model...")
    tokenizer = EsmTokenizer.from_pretrained(ESM_PATH)
    esm_model = AutoModel.from_pretrained(ESM_PATH)
    
    # AutoModelForMaskedLM include encoder + lm_head
    # We only train LM_head
    full_model = AutoModelForMaskedLM.from_pretrained(ESM_PATH)
    decoder = full_model.lm_head.to(DEVICE)
    decoder.train()

    csv_file = "data/antimicrobial.csv" 
    if not os.path.exists(csv_file):
        print(f"Error: {csv_file} not found.")
        return
        
    print(f"Loading Data from {csv_file}...")
    ds = DecoderDataset(csv_file, tokenizer, esm_model, DEVICE)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    optimizer = optim.AdamW(decoder.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

    print(f"Starting Decoder Training with Noise Level = {NOISE_LEVEL}...")

    for epoch in range(EPOCHS):
        total_loss = 0
        pbar = tqdm(dl, desc=f"Epoch {epoch+1}/{EPOCHS}")
        
        for embeddings, labels in pbar:
            embeddings = embeddings.to(DEVICE)
            labels = labels.to(DEVICE)
            
            # --- Noise add ---
            noise = torch.randn_like(embeddings) * NOISE_LEVEL
            
            noisy_embeddings = embeddings + noise
            
            logits = decoder(noisy_embeddings)
            
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix(loss=loss.item())
            
        avg_loss = total_loss / len(dl)
        print(f"Epoch {epoch+1} Avg Loss: {avg_loss:.4f}")
        
    torch.save(decoder.state_dict(), SAVE_PATH)
    print(f"Done! Saved robust decoder to {SAVE_PATH}")
    print("Please update your sample script to use this new decoder path.")

if __name__ == "__main__":
    main()

Loading ESM Model...


Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading Data from data/antimicrobial.csv...
Starting Decoder Training with Noise Level = 0.2...


Epoch 1/5: 100%|█████████████████| 353/353 [00:47<00:00,  7.46it/s, loss=0.0268]


Epoch 1 Avg Loss: 0.2262


...

Epoch 3 Avg Loss: 0.0069


Epoch 4/5: 100%|██████████████████| 353/353 [00:48<00:00,  7.26it/s, loss=0.002]


Epoch 4 Avg Loss: 0.0035


Epoch 5/5: 100%|████████████████| 353/353 [00:49<00:00,  7.06it/s, loss=0.00315]

Epoch 5 Avg Loss: 0.0021
Done! Saved robust decoder to trained_models_cfg/decoder_robust.pth
Please update your sample script to use this new decoder path.
